In [ ]:
# 1. Import Libraries
import pandas as pd
import re
import string
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# Download required NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/trum_tweet_sentiment_analysis.csv")

In [ ]:
# Show first few rows
print("First 5 rows of dataset:")
print(df.head())


First 5 rows of dataset:
                                                text  Sentiment
0  RT @JohnLeguizamo: #trump not draining swamp b...          0
1  ICYMI: Hackers Rig FM Radio Stations To Play A...          0
2  Trump protests: LGBTQ rally in New York https:...          1
3  "Hi I'm Piers Morgan. David Beckham is awful b...          0
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...          0


In [ ]:
# Check column names
print("\nColumns in dataset:", df.columns.tolist())


Columns in dataset: ['text', 'Sentiment']


In [ ]:
df = df[['text', 'Sentiment']]

In [ ]:
# Remove missing values
df.dropna(subset=['text', 'Sentiment'], inplace=True)

In [ ]:
print("\nDataset Shape:", df.shape)


Dataset Shape: (1850123, 2)


In [ ]:
# ============================================
# 4. Text Preprocessing
# ============================================

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtag symbol
    text = re.sub(r'#', '', text)

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords and apply lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    # Join tokens back into sentence
    return " ".join(tokens)

In [ ]:
# Apply preprocessing
df['cleaned_text'] = df['text'].apply(preprocess_text)

print("\nSample Cleaned Text:")
print(df[['text', 'cleaned_text']].head())


Sample Cleaned Text:
                                                text  \
0  RT @JohnLeguizamo: #trump not draining swamp b...   
1  ICYMI: Hackers Rig FM Radio Stations To Play A...   
2  Trump protests: LGBTQ rally in New York https:...   
3  "Hi I'm Piers Morgan. David Beckham is awful b...   
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...   

                                        cleaned_text  
0  rt trump draining swamp taxpayer dollar trip a...  
1  icymi hacker rig fm radio station play antitru...  
2    trump protest lgbtq rally new york bbcworld via  
3  hi im pier morgan david beckham awful donald t...  
4  rt tech firm suing buzzfeed publishing unverif...  


In [ ]:
# ============================================
# 5. Define Features and Labels
# ============================================

X = df['cleaned_text']
y = df['Sentiment']

In [ ]:
# ============================================
# 6. Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


Training Samples: 1480098
Testing Samples: 370025


In [ ]:
# ============================================
# 7. TF-IDF Vectorization
# ============================================

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF Train Shape:", X_train_tfidf.shape)
print("TF-IDF Test Shape:", X_test_tfidf.shape)


TF-IDF Train Shape: (1480098, 5000)
TF-IDF Test Shape: (370025, 5000)


In [ ]:
# ============================================
# 8. Model Training
# ============================================

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
# ============================================
# 9. Prediction
# ============================================

y_pred = model.predict(X_test_tfidf)

In [ ]:
# ============================================
# 10. Evaluation
# ============================================

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy Score:
0.9252239713532869

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.94    248842
           1       0.90      0.86      0.88    121183

    accuracy                           0.93    370025
   macro avg       0.92      0.91      0.91    370025
weighted avg       0.92      0.93      0.92    370025



The model achieved 92.52% accuracy, which indicates strong performance in classifying tweet sentiment.
It performed slightly better on class 0 than class 1, but overall the precision, recall, and F1-scores show that the classifier is effective and reliable.